In [0]:
%run ./00_setup

In [0]:
# Generate Orders data
# Generates random number of orders depending of the time of the day

from pyspark.sql import Window, functions as F
from pyspark.sql.types import StringType
import uuid
import pytz
from datetime import datetime
import random

tz = pytz.timezone('America/New_York')
def get_rand_num_orders():
    # Get the current hour (0-23)
    current_time_est = datetime.now(tz)
    current_hour = current_time_est.hour

    if 0 <= current_hour < 6:
        return 0
    elif 6 <= current_hour < 9:
        return random.randint(0,10)
    elif 9 <= current_hour < 15:
        return random.randint(0,5)
    else:
        return random.randint(0,50)
    
# 1. Get the number of orders (Your existing Python logic is fine here)
num_orders = get_rand_num_orders()
display(num_orders)

def generate_orders_spark(num_orders):
    if num_orders == 0:
        # Returns empty DF with correct schema
        return spark.createDataFrame([], "order_id string, status string, updated_timestamp timestamp, orderdate date, customer_id string")

    # 2. Create the base orders (Parallelized)
    # We generate the metadata for the orders first
    base_orders = spark.range(0, num_orders).select(
        F.expr("uuid()").alias("order_id"),
        F.expr("date_sub(current_date(), cast(rand() * 3 as int))").alias("orderdate"),
        F.lit("Initial").alias("status"),
        F.current_timestamp().alias("updated_timestamp"),
        F.row_number().over(Window.orderBy(F.monotonically_increasing_id())).alias("join_key")
    )

    # 3. Get a random sample of customers from the millions available
    # We only need as many customers as we have orders
    # We use a window to give them a matching join_key
    customers = (spark.table("workspace.amazon_fullfilment_silver.customer_silver")
                 .select("customer_id")
                 .sample(withReplacement=False, fraction=0.1) # Sample 10% to speed up shuffle
                 .withColumn("cust_rand", F.rand())
                 .withColumn("join_key", F.row_number().over(Window.orderBy("cust_rand")))
                 .filter(F.col("join_key") <= num_orders))

    # 4. Join on the incrementing key
    # This is a highly efficient 1-to-1 join
    final_orders = (base_orders.join(customers, "join_key", "inner")
                    .drop("join_key", "cust_rand"))

    return final_orders

# Generate and view
orders_df_raw = generate_orders_spark(num_orders)
orders_df = orders_df_raw.select("order_id", "customer_id" , "orderdate", "status", "updated_timestamp")
#display(orders_df)


#save to the source volume
(orders_df.write
.format("csv")
.option("header",True)
.option("delimiter",",")
.mode("append")
.save(f"{source_volume_path}/orders"))